# ChuckleNet — FINAL Colab Pipeline v20 (2026-09-08)
**Labels: VTT caption laughter markers (Tier-1 weak) · Features: WavLM 768 + prosody 23 · Eval: 5-fold GroupKFold (video-level) F1 + IoU-F1@0.2**

## HOW TO RUN
1. **Runtime → Change runtime type → GPU (T4)** — then *Run all*.
2. Data is already on your Drive: `MyDrive/chuckle_net_1000/` (621 m4a + vtt/). Nothing to upload.
3. **FIRST RUN = GATE MODE** (`GATE_N = 20` below): processes only 20 videos (~5 min) to verify the pipeline end-to-end.
4. If gate passes (features extracted, pos_rate sane, IoU cell runs): set `GATE_N = 0` and *Run all* again → full 621-video run (~2-3 hrs on T4). Checkpoints resume automatically.
5. Results land in `MyDrive/chuckle_net_results/`: `utterance_features_v20.npz` (with per-utterance timestamps), `fusion_model_final.pt`, `results_v20.json`.

## HARD RULES BUILT IN
- **pos_rate < 10% → HARD FAIL** (the Sep-5 disaster: 1.6% positive labels produced garbage)
- **Fresh checkpoint namespace** (`*_v20`) — never loads stale old-format checkpoints
- **Video-level splits only** (GroupKFold on video IDs) — utterance leakage is banned
- **Anchor rule:** these are caption-marker numbers. Report them as such; gold claims (Gillick/StandUp4AI) need separate anchor evals.
- **NO DELETES:** checkpoints/results are additive; nothing is overwritten except within `*_v20` namespace.


In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive')

# Install ffmpeg FIRST for fast m4a decoding
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
!pip install -q soundfile librosa numpy pandas scikit-learn torch transformers tqdm

import numpy as np
import glob
from tqdm import tqdm
import torch
import soundfile as sf
import librosa
import re

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

BASE = '/content/drive/MyDrive/chuckle_net_1000'
AUDIO_DIR = f'{BASE}/audio'
VTT_DIR = f'{BASE}/vtt'
OUTPUT = '/content/drive/MyDrive/chuckle_net_output'
os.makedirs(OUTPUT, exist_ok=True)

audio_files = sorted(glob.glob(f'{AUDIO_DIR}/*.m4a'))
print(f'Audio files: {len(audio_files)}')
print(f'Output dir: {OUTPUT}')

# === v20 CONFIG ===
GATE_N = 20   # 20 = gate mode (first run). Set to 0 for FULL 621-video run.
OUTPUT = '/content/drive/MyDrive/chuckle_net_results'
import os as _os; _os.makedirs(OUTPUT, exist_ok=True)
print(f'GATE_N={GATE_N} | OUTPUT={OUTPUT}')


In [ ]:
# === VERIFY AUDIO LOADING (tests the REAL loader) ===

import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('ffmpeg:', result.stdout.split('\n')[0])

def _test_load(path):
    '''Decode via ffmpeg -> wav in memory. m4a-safe (libsndfile cannot read AAC).'''
    import wave
    from io import BytesIO
    cmd = ['ffmpeg', '-y', '-i', path, '-ar', '16000', '-ac', '1', '-f', 'wav', '-']
    r = subprocess.run(cmd, capture_output=True)
    if r.returncode != 0 or len(r.stdout) == 0:
        return None, None
    with wave.open(BytesIO(r.stdout), 'rb') as w:
        sr = w.getframerate()
        raw = w.readframes(w.getnframes())
    y = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    return y, sr

if audio_files:
    ok = 0
    for test_audio in audio_files[:3]:
        y, sr = _test_load(test_audio)
        name = os.path.basename(test_audio)
        if y is not None and len(y) > 0:
            print(f'  LOAD OK: {name} ({len(y)/sr:.1f}s @ {sr}Hz)')
            ok += 1
        else:
            print(f'  LOAD FAILED: {name}')
    print(f'\nLoader check: {ok}/3 files decoded. If 0/3, STOP — do not run extraction.')
else:
    print('No audio files found — fix BASE path in Cell 1.')


In [ ]:
# === VTT PARSING ===

def parse_timestamp(ts):
    ts = ts.strip().replace(',', '.')
    parts = ts.split(':')
    if len(parts) == 3:
        h, m, s = parts
        return float(h)*3600 + float(m)*60 + float(s)
    elif len(parts) == 2:
        m, s = parts
        return float(m)*60 + float(s)
    return float(ts)

def parse_vtt_for_laughter(vtt_path):
    import re
    try:
        with open(vtt_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        utterances = []
        has_laughter = False
        
        for line in content.split('\n'):
            line = line.strip()
            if '-->' in line:
                if utterances:
                    utterances[-1]['has_laughter'] = has_laughter
                parts = line.split('-->')
                start = parse_timestamp(parts[0])
                end = parse_timestamp(parts[1].strip().split()[0])
                utterances.append({'start': start, 'end': end, 'text': [], 'has_laughter': False})
                has_laughter = False
            elif '[laughter]' in line.lower():
                has_laughter = True
            elif re.search(r'\[(/?(laughter|laugh|laughing|lol|mdr))\]', line, re.IGNORECASE):
                has_laughter = True
            elif utterances and not line.startswith('<'):
                clean = re.sub(r'<[^>]+>', '', line)
                if clean.strip():
                    utterances[-1]['text'].append(clean)
        
        if utterances:
            utterances[-1]['has_laughter'] = has_laughter
        return utterances
    except Exception as e:
        print(f'VTT error: {e}')
        return None

In [ ]:
# === AUDIO LOADING (ffmpeg subprocess — the PROVEN m4a path) ===

import subprocess
import wave
from io import BytesIO

def load_audio(audio_path, sr_target=(16000, 22050)):
    '''Decode ANY format via ffmpeg to 16k mono wav, then resample for 22k.
    NOTE: soundfile/libsndfile CANNOT decode m4a/AAC; librosa>=0.10 has no
    audioread fallback. ffmpeg subprocess is the reliable path in Colab.'''
    try:
        cmd = ['ffmpeg', '-y', '-i', audio_path, '-ar', '16000', '-ac', '1', '-f', 'wav', '-']
        r = subprocess.run(cmd, capture_output=True)
        if r.returncode != 0 or len(r.stdout) == 0:
            return None
        with wave.open(BytesIO(r.stdout), 'rb') as w:
            sr16 = w.getframerate()
            raw = w.readframes(w.getnframes())
        y_16k = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
        if len(y_16k) < sr16:
            return None
        result = {16000: y_16k}
        for sr_t in sr_target:
            if sr_t != 16000:
                result[sr_t] = librosa.resample(y_16k, orig_sr=16000, target_sr=sr_t)
        return result
    except Exception as e:
        print(f'Audio error: {e}')
        return None


In [ ]:
# === PROSODY EXTRACTION ===

def extract_prosody_segment(y, sr, start_s, end_s, hop_length=512):
    start_sample = int(start_s * sr)
    end_sample = int(end_s * sr)
    y_seg = y[start_sample:end_sample]
    if len(y_seg) < sr * 0.1:
        return None
    
    features = []
    rms = librosa.feature.rms(y=y_seg, hop_length=hop_length)[0]
    features.extend([np.mean(rms), np.std(rms), np.max(rms)])
    
    f0, voiced, prob = librosa.pyin(y_seg, fmin=50, fmax=500, sr=sr, hop_length=hop_length)
    f0 = np.nan_to_num(f0, nan=0)
    f0_valid = f0[f0 > 0]
    if len(f0_valid) > 0:
        features.extend([np.mean(f0_valid), np.std(f0_valid), np.max(f0_valid)-np.min(f0_valid)])
    else:
        features.extend([0, 0, 0])
    
    zcr = librosa.feature.zero_crossing_rate(y_seg, hop_length=hop_length)[0]
    features.extend([np.mean(zcr), np.std(zcr)])
    
    sc = librosa.feature.spectral_centroid(y=y_seg, sr=sr, hop_length=hop_length)[0]
    features.extend([np.mean(sc), np.std(sc)])
    
    sb = librosa.feature.spectral_bandwidth(y=y_seg, sr=sr, hop_length=hop_length)[0]
    features.extend([np.mean(sb), np.std(sb)])
    
    rolloff = librosa.feature.spectral_rolloff(y=y_seg, sr=sr, hop_length=hop_length)[0]
    features.extend([np.mean(rolloff), np.std(rolloff)])
    
    mfcc = librosa.feature.mfcc(y=y_seg, sr=sr, n_mfcc=13, hop_length=hop_length)
    for i in range(13):
        features.append(np.mean(mfcc[i]))
    
    return np.array(features, dtype=np.float32)

In [ ]:
# === WAVLM SETUP ===

from transformers import Wav2Vec2Model

print('Loading WavLM...')
wavlm = Wav2Vec2Model.from_pretrained('microsoft/wavlm-base')
wavlm.to(DEVICE)
wavlm.eval()
print(f'WavLM loaded on {DEVICE}')

def extract_wavlm_segment(y_16k, start_s, end_s):
    start_sample = int(start_s * 16000)
    end_sample = int(end_s * 16000)
    y_seg = y_16k[start_sample:end_sample]
    if len(y_seg) < 1600:
        return None
    with torch.no_grad():
        inputs = torch.FloatTensor(y_seg).unsqueeze(0).to(DEVICE)
        am = torch.ones_like(inputs)
        out = wavlm(inputs, attention_mask=am)
        emb = out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return emb.astype(np.float32)

In [ ]:
# === MAIN EXTRACTION ===

if GATE_N and GATE_N > 0:
    audio_files = audio_files[:GATE_N]
    print(f'GATE MODE: processing first {GATE_N} videos only')

CHECKPOINT_FILE = f'{OUTPUT}/extraction_checkpoint_v20.npz'
CHECKPOINT_IDX = f'{OUTPUT}/processed_idx_v20.txt'

processed_idx = set()
if os.path.exists(CHECKPOINT_IDX):
    with open(CHECKPOINT_IDX, 'r') as f:
        processed_idx = set(f.read().splitlines())
    print(f'Resuming from {len(processed_idx)} files')

def save_checkpoint(features, labels, vids, starts, ends):
    np.savez_compressed(CHECKPOINT_FILE, features=features, labels=labels, vids=np.array(vids),
                        utt_starts=np.array(starts), utt_ends=np.array(ends))
    with open(CHECKPOINT_IDX, 'w') as f:
        f.write('\n'.join(sorted(processed_idx)))
    print(f'Checkpoint: {len(features)} samples at {len(processed_idx)} files')

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        data = np.load(CHECKPOINT_FILE, allow_pickle=True)
        if len(data['features']) > 0:
            st = list(data['utt_starts']) if 'utt_starts' in data else [None]*len(data['features'])
            en = list(data['utt_ends']) if 'utt_ends' in data else [None]*len(data['features'])
            return data['features'], data['labels'], list(data['vids']), st, en
    return None, None, None, None, None

start_idx = 0
if processed_idx:
    for i, af in enumerate(audio_files):
        vid = os.path.basename(af).replace('.m4a', '')
        if vid not in processed_idx:
            start_idx = i
            break
    else:
        print(f'All {len(audio_files)} files done!')

print(f'Starting from index {start_idx}/{len(audio_files)}')

# Pre-flight check: abort loudly if no audio files found
if not audio_files:
    raise RuntimeError(f'ERROR: audio_files is empty! Check that Cell 1 ran and data exists at {AUDIO_DIR}.')
print(f'Processing {len(audio_files)} audio files...')


all_features, all_labels, all_vids = [], [], []
all_starts, all_ends = [], []
stats = {'no_vtt': 0, 'no_audio': 0, 'no_utt': 0, 'bad_feat': 0}
existing = load_checkpoint()
# Load checkpoint whenever resume mode (processed_idx non-empty).
# A truly fresh run has empty processed_idx -> no stale data loaded.
if existing[0] is not None and len(existing[0]) > 0 and len(processed_idx) > 0:
    all_features, all_labels, all_vids = list(existing[0]), list(existing[1]), list(existing[2])
    all_starts, all_ends = list(existing[3]), list(existing[4])
    print(f'Resuming with {len(all_features)} samples from checkpoint')

for i, af in enumerate(tqdm(audio_files[start_idx:], desc='Processing')):
    vid = os.path.basename(af).replace('.m4a', '')
    if vid in processed_idx:
        continue
    
    vtt_path = None
    for p in [f'{VTT_DIR}/{vid}.vtt', f'{VTT_DIR}/{vid}.en.vtt', f'{VTT_DIR}/{vid}.en-US.vtt']:
        if os.path.exists(p):
            vtt_path = p
            break
    if not vtt_path:
        matches = glob.glob(f'{VTT_DIR}/{vid}*.vtt')
        if matches:
            vtt_path = matches[0]
    if not vtt_path:
        stats['no_vtt'] += 1
        continue
    
    utterances = parse_vtt_for_laughter(vtt_path)
    if not utterances:
        stats['no_utt'] += 1
        continue
    
    audio_data = load_audio(af)
    if audio_data is None:
        stats['no_audio'] += 1
        continue
    
    y_16k = audio_data[16000]
    y_22k = audio_data[22050]
    
    for utt in utterances:
        prosody = extract_prosody_segment(y_22k, 22050, utt['start'], utt['end'])
        wavlm_emb = extract_wavlm_segment(y_16k, utt['start'], utt['end'])
        if prosody is not None and wavlm_emb is not None:
            all_features.append(np.concatenate([wavlm_emb, prosody]))
            all_labels.append(1 if utt['has_laughter'] else 0)
            all_vids.append(vid)
            all_starts.append(float(utt['start'])); all_ends.append(float(utt['end']))
        else:
            stats['bad_feat'] += 1
    
    processed_idx.add(vid)
    if len(processed_idx) % 20 == 0:
        save_checkpoint(np.array(all_features), np.array(all_labels), all_vids, all_starts, all_ends)

if all_features:
    save_checkpoint(np.array(all_features), np.array(all_labels), all_vids, all_starts, all_ends)

print(f'\nExtracted: {len(all_features)} samples')
y_arr = np.array(all_labels)
if len(y_arr) > 0:
    print(f'Positive: {y_arr.sum()} ({100*y_arr.mean():.1f}%)')
print(f'Skipped — no_vtt:{stats["no_vtt"]}, no_audio:{stats["no_audio"]}, no_utt:{stats["no_utt"]}, bad_feat:{stats["bad_feat"]}')

# === v20 HARD RULE: pos_rate gate ===
y_arr = np.array(all_labels)
pos_rate = float(y_arr.mean()) if len(y_arr) else 0.0
if pos_rate < 0.10:
    raise RuntimeError(f'HARD FAIL: positive rate {pos_rate*100:.1f}% < 10% — labels are broken '
                       f'(cf. Sep-5 checkpoint disaster: 1.6% pos). DO NOT TRAIN. Check VTT parsing / video mix.')
print(f'pos_rate gate PASSED: {pos_rate*100:.1f}%')


In [ ]:
# === SAVE FEATURES ===

if not all_features:
    raise RuntimeError('ERROR: No features extracted! Check skip stats above.')

np.savez_compressed(f'{OUTPUT}/utterance_features_v20.npz',
                    features=np.array(all_features),
                    labels=np.array(all_labels),
                    vids=np.array(all_vids),
                    utt_starts=np.array(all_starts),
                    utt_ends=np.array(all_ends))
print(f'Saved: {OUTPUT}/utterance_features_v20.npz ({len(all_features)} samples)')


In [ ]:
# === FUSION MLP TRAINING ===

import torch
import torch.nn as nn
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

import random
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

data = np.load(f'{OUTPUT}/utterance_features_v20.npz')
X, y, vids = data['features'], data['labels'], data['vids']

print(f'Data: {len(y)} samples, {len(set(vids))} videos')
print(f'Features: {X.shape}')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

class FusionMLP(nn.Module):
    def __init__(self, dim=789, hidden=[512, 256, 64]):
        super().__init__()
        self.bn0 = nn.BatchNorm1d(dim)
        layers = []
        prev = dim
        for h in hidden:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(self.bn0(x)).squeeze(-1)

gkf = GroupKFold(n_splits=5)
f1s, precs, recs = [], [], []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_s, y, vids)):
    X_tr, y_tr = torch.FloatTensor(X_s[tr_idx]), torch.FloatTensor(y[tr_idx])
    X_te, y_te = torch.FloatTensor(X_s[te_idx]), y[te_idx]
    
    model = FusionMLP(dim=X_s.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    pos_weight = torch.tensor([(1-y_tr.mean())/max(0.01, y_tr.mean())]).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    for epoch in range(100):
        model.train()
        n_tr = len(X_tr) - 1 if len(X_tr) % 32 == 1 else len(X_tr)  # Batch1d crashes on batch size 1
        for i in range(0, n_tr, 32):
            batch_x = X_tr[i:i+32].to(DEVICE)
            batch_y = y_tr[i:i+32].to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(batch_x), batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
    
    model.eval()
    with torch.no_grad():
        preds = torch.sigmoid(model(X_te.to(DEVICE))).cpu().numpy()
        pred_binary = (preds > 0.5).astype(int)
        f1 = f1_score(y_te, pred_binary, zero_division=0)
        prec = precision_score(y_te, pred_binary, zero_division=0)
        rec = recall_score(y_te, pred_binary, zero_division=0)
        f1s.append(f1); precs.append(prec); recs.append(rec)
    print(f'Fold {fold+1}: F1={f1:.4f}, P={prec:.4f}, R={rec:.4f}')

print(f'\nMean F1: {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}')

In [ ]:
# === SAVE MODEL ===

torch.save({'model_state_dict': model.state_dict(), 'scaler': scaler,
            'f1_mean': np.mean(f1s), 'f1_std': np.std(f1s)},
           f'{OUTPUT}/fusion_model_final.pt')
print(f'Model saved: {OUTPUT}/fusion_model_final.pt')

In [ ]:
# === v20: IoU-F1@0.2 EVALUATION (the honest interval metric) ===
import numpy as np, torch, json
from sklearn.model_selection import GroupKFold
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

data = np.load(f'{OUTPUT}/utterance_features_v20.npz')
X, y, vids = data['features'], data['labels'], data['vids']
starts, ends = data['utt_starts'], data['utt_ends']
scaler = StandardScaler(); X_s = scaler.fit_transform(X)

class FusionMLP(nn.Module):
    def __init__(self, dim, hidden=[512,256,64]):
        super().__init__()
        self.bn0 = nn.BatchNorm1d(dim); layers=[]; prev=dim
        for h in hidden:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]; prev=h
        layers.append(nn.Linear(prev,1)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(self.bn0(x)).squeeze(-1)

def merge_iv(ivs, gap=0.8):
    if not ivs: return []
    ivs = sorted(ivs); out=[list(ivs[0])]
    for s,e in ivs[1:]:
        if s - out[-1][1] <= gap: out[-1][1] = max(out[-1][1], e)
        else: out.append([s,e])
    return [(s,e) for s,e in out]

def iou_f1(vid_list, y_true, y_pred, starts_, ends_, thr=0.2):
    """Per-video interval matching F1 at IoU thr (greedy by IoU)."""
    tps = fps = fns = 0
    for v in set(vid_list):
        idx = [i for i,vv in enumerate(vid_list) if vv==v]
        t_iv = merge_iv([(starts_[i], ends_[i]) for i in idx if y_true[i]==1])
        p_iv = merge_iv([(starts_[i], ends_[i]) for i in idx if y_pred[i]==1])
        matched_p, matched_t = set(), set()
        pairs = sorted((( iou(a,b), j, k) for j,a in enumerate(t_iv) for k,b in enumerate(p_iv)), reverse=True)
        for val, j, k in pairs:
            if val < thr: break
            if j not in matched_t and k not in matched_p:
                matched_t.add(j); matched_p.add(k); tps += 1
        fps += len(p_iv) - len(matched_p); fns += len(t_iv) - len(matched_t)
    prec = tps/max(1,tps+fps); rec = tps/max(1,tps+fns)
    return 2*prec*rec/max(1e-9,prec+rec), prec, rec

gkf = GroupKFold(n_splits=5)
seg_f1s, ious = [], []
torch.manual_seed(42); np.random.seed(42)
for fold,(tr,te) in enumerate(gkf.split(X_s, y, vids)):
    m = FusionMLP(X_s.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    Xtr = torch.FloatTensor(X_s[tr]); ytr = torch.FloatTensor(y[tr])
    pw = torch.tensor([(1-ytr.mean())/max(0.01,ytr.mean())]).to(DEVICE)
    lf = nn.BCEWithLogitsLoss(pos_weight=pw)
    n_tr = len(Xtr)-1 if len(Xtr)%32==1 else len(Xtr)
    for ep in range(100):
        m.train()
        for i in range(0, n_tr, 32):
            bx, by = Xtr[i:i+32].to(DEVICE), ytr[i:i+32].to(DEVICE)
            opt.zero_grad(); loss = lf(m(bx), by); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step()
    m.eval()
    with torch.no_grad():
        pr = (torch.sigmoid(m(torch.FloatTensor(X_s[te]).to(DEVICE))).cpu().numpy() > 0.5).astype(int)
    f1v = iou_f1(list(np.array(vids)[te]), y[te], pr, np.array(starts)[te], np.array(ends)[te])
    ious.append(f1v[0])
    print(f'Fold {fold+1}: IoU-F1@0.2 = {f1v[0]:.4f} (P={f1v[1]:.3f} R={f1v[2]:.3f})')
print(f'\nMEAN IoU-F1@0.2: {np.mean(ious):.4f} +/- {np.std(ious):.4f}   (reference: naive-baseline ~0.29, StandUp4AI 0.51)')

In [ ]:
# === v20: RESULTS JSON (append-only provenance record) ===
import datetime, sys, transformers
results = {
  'schema': 'chucklenet_v20', 'timestamp_utc': datetime.datetime.utcnow().isoformat(),
  'label_tier': 'TIER-1 weak: VTT caption laughter markers ([laughter]/[laugh]/[lol]/[mdr]) — report as caption-marker detection, NOT gold laughter detection',
  'anchor_rule': 'Gold claims require separate Gillick-162v / StandUp4AI anchor evals (docs/LABEL_HIERARCHY.md)',
  'config': {'gate_n': GATE_N, 'features': 'WavLM-base-768 + prosody-23 (795-dim)', 'model': 'FusionMLP 512-256-64, seed 42',
             'split': 'GroupKFold 5 (video-level)', 'pos_rate': float(np.array(all_labels).mean()),
             'n_samples': int(len(all_labels)), 'n_videos': int(len(set(all_vids)))},
  'segment_f1': {'mean': float(np.mean(f1s)), 'std': float(np.std(f1s)), 'folds': [float(x) for x in f1s]},
  'iou_f1_02': {'mean': float(np.mean(ious)), 'std': float(np.std(ious)), 'folds': [float(x) for x in ious],
                'merge_gap_s': 0.8, 'note': 'merge 0.8 = BEST_118V best config'},
  'env': {'torch': torch.__version__, 'transformers': transformers.__version__, 'python': sys.version.split()[0]},
  'references': {'naive_baseline_IoU': 0.29, 'standup4ai_baseline': 0.51, 'gillick_gold_segment_f1': 0.559}
}
with open(f'{OUTPUT}/results_v20.json','w') as f: json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))
print(f'\nSaved: {OUTPUT}/results_v20.json — BACK THIS UP (no-delete policy)')